In [11]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3


load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [12]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]

In [13]:
from sqlalchemy import create_engine, text

DB_URL = "sqlite:///prices.db"
engine = create_engine(DB_URL)

def get_ticket_price(city:str) -> str:
    city = city.lower()
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    query = "SELECT price FROM prices WHERE city = :city"

    with engine.connect() as conn:
        result = conn.execute(text(query), {"city":city})
        row = result.fetchone()

    return str(row[0]) if row else "there's no city in db"

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content": price_details,
                "tool_call_id":tool_call.id
            })
    return responses

def chat(user_message, history):
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + formatted_history + [{"role": "user", "content": user_message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)
        
        messages.append(assistant_message)
        messages.extend(tool_responses)
        
        # Call OpenAI again with the new tool context
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [15]:
import base64
from io import BytesIO
from PIL import Image

In [16]:
def artist(city:str):
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"""
        An image representing a vacation in {city},
        showing tourist spots and everything unique about {city}, in a vibrant pop-art style""",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [17]:
# image = artist("New York City")
# display(image)

In [18]:
from openai.types.chat.chat_completion_message import ChatCompletionMessage

def handle_tool_calls_and_return_cities(message:ChatCompletionMessage):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })
    return responses, cities

In [19]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    if cities:
        image = artist(cities[0])
    
    return history, image


In [21]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
